# FLUX.1-dev LoRA Training for Sprites
This notebook uses **ai-toolkit** to fine-tune FLUX.1-dev on your sprite dataset.
Optimized for Kaggle T4 (16GB VRAM) using NF4 quantization and Gradient Checkpointing.

In [ ]:
# Step 1: Install Dependencies
!pip install -q torch torchvision accelerate transformers diffusers peft bitsandbytes safetensors sentencepiece
!git clone https://github.com/ostris/ai-toolkit.git /kaggle/working/ai-toolkit
%cd /kaggle/working/ai-toolkit
!git submodule update --init --recursive
!pip install -q -r requirements.txt
import sys
sys.path.insert(0, '/kaggle/working/ai-toolkit')

In [ ]:
# Step 2: Prepare Dataset from HF
import os
from datasets import load_dataset
from tqdm import tqdm
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except:
    HF_TOKEN = ''

login(token=HF_TOKEN)

DATASET_ID = 'darklord8777/sprites'
SAVE_DIR = '/kaggle/working/train_data'
os.makedirs(SAVE_DIR, exist_ok=True)

ds = load_dataset(DATASET_ID, split='train')
print(f'Downloading {len(ds)} images...')

for i, item in enumerate(tqdm(ds)):
    img = item['image'].convert('RGB')
    img_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.png')
    txt_path = os.path.join(SAVE_DIR, f'sprite_{i:04d}.txt')
    
    img.save(img_path)
    
    # Auto-captioning based on your metadata
    cls = item.get('class', 'character')
    act = item.get('action', 'idle')
    dir = item.get('direction', 'front')
    caption = f'a pixel art sprite of a {cls}, {act} pose, {dir} view, high quality, transparent background'
    
    with open(txt_path, 'w') as f:
        f.write(caption)

In [ ]:
# Step 3: Generate YAML Config for 16GB VRAM
import yaml

config = {
    'job': 'extension',
    'config': {
        'name': 'sprite_flux_lora',
        'process': [{
            'type': 'sd_trainer',
            'training_folder': '/kaggle/working/output',
            'device': 'cuda:0',
            'model': {
                'name_or_path': 'black-forest-labs/FLUX.1-dev',
                'is_flux': True,
                'quantize': True  # Required for 16GB
            },
            'network': {
                'type': 'lora',
                'linear': 16,
                'linear_alpha': 16
            },
            'train': {
                'batch_size': 1,
                'gradient_accumulation_steps': 4,
                'gradient_checkpointing': True,
                'steps': 2000,
                'lr': 0.0001,
                'optimizer': 'adamw8bit',
                'dtype': 'bf16'
            },
            'datasets': [{
                'folder_path': '/kaggle/working/train_data',
                'caption_ext': 'txt',
                'resolution': [512]
            }],
            'save': {
                'push_to_hub': True,
                'hf_repo_id': 'darklord8777/sprite-generator-model',
                'hf_private': True,
                'save_every': 500
            },
            'sample': {
                'sampler': 'flowmatch',
                'sample_every': 250,
                'prompts': [
                    'a pixel art sprite of a character, idle pose, front view',
                    'a pixel art sprite of an enemy, attack pose, side view'
                ]
            }
        }]
    }
}

with open('/kaggle/working/config.yaml', 'w') as f:
    yaml.dump(config, f)

In [ ]:
# Step 4: Start Training
!python run.py /kaggle/working/config.yaml